<a href="https://colab.research.google.com/github/npeyton/Dostoyevsky_GPT/blob/main/Dostoyevsky_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request
import math
torch.manual_seed(1337)

In [45]:

#loading in the data and taking out headers
crime_url = 'https://raw.githubusercontent.com/npeyton/Dostoyevsky_GPT/main/crime_and_punishment.txt'
crime = urllib.request.urlopen(crime_url).read().decode('utf-8')
crime = crime[6388:]

brothers_url = 'https://raw.githubusercontent.com/npeyton/Dostoyevsky_GPT/main/karamazov.txt'
brothers = urllib.request.urlopen(brothers_url).read().decode('utf-8')
brothers = brothers[5200:]

white_url = 'https://raw.githubusercontent.com/npeyton/Dostoyevsky_GPT/main/white_nights.txt'
white = urllib.request.urlopen(white_url).read().decode('utf-8')
white = white[2005:]

print(crime[:1000])






CRIME AND PUNISHMENT




PART I


CHAPTER I


On an exceptionally hot evening early in July a young man came out of
the garret in which he lodged in S. Place and walked slowly, as though
in hesitation, towards K. bridge.

He had successfully avoided meeting his landlady on the staircase.
His garret was under the roof of a high, five-storied house and was
more like a cupboard than a room. The landlady who provided him with
garret, dinners, and attendance, lived on the floor below, and every
time he went out he was obliged to pass her kitchen, the door of which
invariably stood open. And each time he passed, the young man had a
sick, frightened feeling, which made him scowl and feel ashamed. He was
hopelessly in debt to his landlady, and was afraid of meeting her.

This was not because he was cowardly and abject, quite the contrary;
but for some time past he had been in an overstrained irritable
condition, verging on hypochondria. He had become so comple


In [46]:
#making the vocabulary / all of the symbols in the data.
chars = sorted(list(set(crime+brothers+white)))
print(len(chars))
vocab_size = len(chars)
#encoding and decoding
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] # encoder: take a string -  output ints
decode = lambda s: [stoi[c] for c in s] # encoder: take a list of ints - output string



110


In [47]:
# Train and test splits
crime_data = torch.tensor(encode(crime), dtype=torch.long)
brothers_data = torch.tensor(encode(brothers), dtype=torch.long)
white_data = torch.tensor(encode(white), dtype=torch.long)

# crime data
n = int(0.9*len(crime_data)) # first 90% will be train, rest val
c_train_data = crime_data[:n]
c_val_data = crime_data[n:]

# brothers data
n = int(0.9*len(brothers_data)) # first 90% will be train, rest val
b_train_data = brothers_data[:n]
b_val_data = brothers_data[n:]

# white data
n = int(0.9*len(white_data)) # first 90% will be train, rest val
w_train_data = white_data[:n]
w_val_data = white_data[n:]

total_train = torch.cat([c_train_data, b_train_data, w_train_data])
total_val = torch.cat([c_val_data, b_val_data, w_val_data])

# this way we can train on each book individually then evaluate the loss.
# but we can also train on all of the books together and see how the total model does.
# and we can also train on only two books and test on the third book to see how that does.
# allowing for some experimentation by paritioning in this way.

Need a cell here so that I can make whatever data I want to use for the model just train and val so that I can use the other code. Essentially here i just want to set the train and val data to whatever is relevant for the current run.

In [48]:
train_data = total_train
val_data = total_val

Take either the training data or the val data, and grab a random position in the data from which we grab a block sized amount of data for the defined amount of batches. Then iterate through that sequence going from just the first character to the block_size character grabing the combinations and making them equal to x and then the next character for each of those sequences equal to y.

In [49]:
# get the batches from the above defined data
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


So for both the training split and for the validation split this code makes a tensor of length (eval_iters) aka the amount of times you want to iterate, and it gets a batch of sequences runs the model on X and Y and calculates the loss and the logits. it stores the losses.mean in the correct vector in the out tensor.


In [50]:
#calculate the loss function
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


ATTENTION MECHANISM:
This is the key of the whole project understanding attention.

Here we have the key query value matrices initialized.

- We have the key matrix aka what do I have
- The query matrix aka what am I looking for.
- Value matrix aka what do I give if someone is looking for me

We take every characters query, compare it to all of the previous characters keys and arrive at the "affinites" which are basically how much each key is affecting the query token. Then softmax that and we arrive at the Value matrix. then with that we multiply it by the weights.  

In [51]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

This allows you to do the above but with multiple heads of attention. aka we are gainging this affinity information for all of the tokens in the same sequence but we are doing it for multiple heads in parallel. We are essentially gaining more contextual information with each head.

In [52]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


One feed foward loop aka a linear layer, then a ReLU, then another linear layer, then dropout


In [53]:
# feed Forward simple linear layer of a NN.
# allows the layer to expand to 4x the size, then summazize back down then dropout.
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [54]:
# making a transformer block
#   multihead attention layer
#   linear layer to digest the information.
#   normalize so the residuals do not blow up and lose neurons from softmax.
#   then add the learned information back to the residual x
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


OK the whole model in one now.
- First embed the token into the embedding dim.
- Then embed the position information into the same dimension.
- Then run a series of transformer blocks (attention, linear, normalize, add back)
- Then a final normalize layer,
- Followed by the last linear layer to transform back out of the embedding dimension to the vocab.
- If targets are provided then we will calculate loss
- Also includes the generate function to generate text.


In [55]:
# all together now


class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return id

In [ ]:

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
# print out the decoded tokens that were generated (setting new tokens to 500 )
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

Using our new model!!

In [56]:
#hyperparameters
batch_size = 32 # how many sequences to process in parallel
block_size = 64 # maximum context length
max_iters = 1000 # total iterations of training
eval_interval = 200 # how often to evaluate the loss.
learning_rate = .1 # size of jump during gradient descent.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0

In [43]:
results = {}

In [21]:
# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


# results[learning_rate] = (losses['train'].item(), losses['val'].item())


0.82827 M parameters
step 0: train loss 4.7434, val loss 4.7421
step 999: train loss 2.5683, val loss 2.6542
{0.1: (3.0201456546783447, 3.0488107204437256), 1: (3.181978464126587, 3.2204978466033936), 0.01: (2.5682976245880127, 2.654207468032837)}


Lets find a good learning rate. Going to go from 0 - 1 on an log scale. testing 10 different learning rates and finding a good one to go with. We want to find a learning rate where the cost (loss) decreases in the first few epochs (iterations). We also want a learning rate that gets close to the lowest loss out of the bunch but to avoid overfitting we will chose one that is slightly less (by a factor of 2 less than)

In [57]:
# potential learning rates
lr_list = []

for i in range (8):
  lr_list.append(10**(-i))

print(lr_list)

[1, 0.1, 0.01, 0.001, 0.0001, 1e-05, 1e-06, 1e-07]


In [58]:
#training the model one time per learning rate to get an understanding of how the loss curve looks for each one
for i in range(len(lr_list)):

  torch.manual_seed(1337) # same start point each time.

  learning_rate = lr_list[i]
  print(f'\n\nNew test - Learning rate: {lr_list[i]}')

  model = GPTLanguageModel()
  m = model.to(device)
  # print the number of parameters in the model
  print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

  # create a PyTorch optimizer
  optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

  # Training the model
  for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    results[learning_rate] = (losses['train'].item(), losses['val'].item())




New test - Learning rate: 1
0.82827 M parameters
step 0: train loss 4.7163, val loss 4.7216
step 200: train loss 298.3237, val loss 299.9781
step 400: train loss 100.4519, val loss 100.2374
step 600: train loss 133.1588, val loss 132.8354
step 800: train loss 159.7744, val loss 158.9921
step 999: train loss 458.9704, val loss 454.3654


New test - Learning rate: 0.1
0.82827 M parameters
step 0: train loss 4.7163, val loss 4.7216
step 200: train loss 3.1912, val loss 3.2126
step 400: train loss 3.1721, val loss 3.1890
step 600: train loss 3.1541, val loss 3.1740
step 800: train loss 3.1466, val loss 3.1759
step 999: train loss 3.1548, val loss 3.1936


New test - Learning rate: 0.01
0.82827 M parameters
step 0: train loss 4.7163, val loss 4.7216
step 200: train loss 2.6152, val loss 2.7095
step 400: train loss 2.5618, val loss 2.6384
step 600: train loss 2.5814, val loss 2.6596
step 800: train loss 2.6017, val loss 2.6712
step 999: train loss 2.6031, val loss 2.6964


New test - Learn

Given the above results we a good learning rate is likely in the range from .0001 to .01 So lets look a bit deeper.

In [59]:
# Deeper dive on learning rates
lr_list = [.0001,.0003,.0006,.001,.003,.006,.01]

print(lr_list)

[0.0001, 0.0003, 0.0006, 0.001, 0.003, 0.006, 0.01]


In [60]:
#training the model one time per learning rate to get an understanding of how the loss curve looks for each one
for i in range(len(lr_list)):

  torch.manual_seed(1337) # same start point each time.

  learning_rate = lr_list[i]
  print(f'\n\nNew test - Learning rate: {lr_list[i]}')

  model = GPTLanguageModel()
  m = model.to(device)
  # print the number of parameters in the model
  print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

  # create a PyTorch optimizer
  optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

  # Training the model
  for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    results[learning_rate] = (losses['train'].item(), losses['val'].item())



New test - Learning rate: 0.0001
0.82827 M parameters
step 0: train loss 4.7163, val loss 4.7216
step 200: train loss 2.7390, val loss 2.7870
step 400: train loss 2.4753, val loss 2.5270
step 600: train loss 2.3013, val loss 2.3855
step 800: train loss 2.1442, val loss 2.2341
step 999: train loss 2.0120, val loss 2.1263


New test - Learning rate: 0.0003
0.82827 M parameters
step 0: train loss 4.7163, val loss 4.7216
step 200: train loss 2.4268, val loss 2.5147
step 400: train loss 2.1911, val loss 2.2801
step 600: train loss 1.9577, val loss 2.0817
step 800: train loss 1.7963, val loss 1.9120
step 999: train loss 1.6998, val loss 1.8407


New test - Learning rate: 0.0006
0.82827 M parameters
step 0: train loss 4.7163, val loss 4.7216
step 200: train loss 2.3249, val loss 2.4355
step 400: train loss 1.9795, val loss 2.0875
step 600: train loss 1.7661, val loss 1.9054
step 800: train loss 1.6550, val loss 1.7824
step 999: train loss 1.5811, val loss 1.7374


New test - Learning rate: 

Ok so it looks like our best learning rate is 0.0006. This should give us a good Starting point to work out the rest of the hyperparameters from.

Next we are going to look at actual Model Size. We are going to manipulate the n_embd aka the embedding dimension, the n_head aka the number of heads of attention. and n_layers aka the number of blocks in the tranformer. For these we are always looking to have the embedding dimension divisible by the heads of attention for efficiency sake.

First new Change is going to be the embedding dimension - Lets go from 128 to 256 to see how that changes things. ( we are also going to run tests for a bit more iterations) Remember we are looking to improve upon the test results below.

Test with Learning rate .0006:
- Step 999: train loss 1.5811, val loss 1.7374

In [66]:
#hyperparameters
batch_size = 32 # how many sequences to process in parallel
block_size = 64 # maximum context length
max_iters = 2000 # total iterations of training
eval_interval = 200 # how often to evaluate the loss.
learning_rate = .0006 # size of jump during gradient descent.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 256
n_head = 4
n_layer = 4
dropout = 0

In [67]:
# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

3.229294 M parameters
step 0: train loss 4.6391, val loss 4.6348
step 200: train loss 2.0881, val loss 2.1767
step 400: train loss 1.7422, val loss 1.8749
step 600: train loss 1.6224, val loss 1.7486
step 800: train loss 1.5543, val loss 1.6937
step 1000: train loss 1.4916, val loss 1.6545
step 1200: train loss 1.4445, val loss 1.5877
step 1400: train loss 1.4137, val loss 1.5770
step 1600: train loss 1.3975, val loss 1.5527
step 1800: train loss 1.3742, val loss 1.5417
step 1999: train loss 1.3547, val loss 1.5285


Ok a bit of an improvement lets keep going by testing n_layer as 6 now.

In [68]:
n_layer = 6

In [69]:
# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f},  Gap {losses['val']-losses['train']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

4.807278 M parameters
step 0: train loss 4.6112, val loss 4.6086
step 200: train loss 2.1287, val loss 2.2261
step 400: train loss 1.7831, val loss 1.9224
step 600: train loss 1.6356, val loss 1.7699
step 800: train loss 1.5523, val loss 1.7277
step 1000: train loss 1.4904, val loss 1.6577
step 1200: train loss 1.4562, val loss 1.5825
step 1400: train loss 1.4200, val loss 1.5628
step 1600: train loss 1.3910, val loss 1.5500
step 1800: train loss 1.3688, val loss 1.5158
step 1999: train loss 1.3590, val loss 1.4756


Ok still getting improvement now lets bump n_embd to 360 and going to also train for a bit longer now because it seems like we are cutting off training a bit early. I would rather see a training run plateau before making judgement on how much improvement we gained from it. Also keeping track of the gap between training and validation just to judge overfitting

In [74]:
n_embd = 360
max_iters = 4000
eval_interval = 500

# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, Gap {losses['val']-losses['train']:.4f} ")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

9.45587 M parameters
step 0: train loss 4.6694, val loss 4.6644, Gap -0.0049 
step 500: train loss 1.9049, val loss 2.0323, Gap 0.1274 
step 1000: train loss 1.5802, val loss 1.7435, Gap 0.1634 
step 1500: train loss 1.4482, val loss 1.6470, Gap 0.1988 
step 2000: train loss 1.3781, val loss 1.5645, Gap 0.1864 
step 2500: train loss 1.3185, val loss 1.4836, Gap 0.1652 
step 3000: train loss 1.2980, val loss 1.4597, Gap 0.1617 
step 3500: train loss 1.2767, val loss 1.4569, Gap 0.1801 
step 3999: train loss 1.2450, val loss 1.4329, Gap 0.1879 


Adding another 2 heads of attention to be 6 now.

So that is 360 n_embd, 6 n_head, and n_layer 6 as well

In [78]:
# lets see how this compares to the above
n_head = 6

# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, Gap {losses['val']-losses['train']:.4f} ")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

9.45587 M parameters
step 0: train loss 4.7472, val loss 4.7434, Gap -0.0038 
step 500: train loss 1.9511, val loss 2.0535, Gap 0.1024 
step 1000: train loss 1.6243, val loss 1.7704, Gap 0.1461 
step 1500: train loss 1.4823, val loss 1.6432, Gap 0.1609 
step 2000: train loss 1.4102, val loss 1.5675, Gap 0.1573 
step 2500: train loss 1.3438, val loss 1.5014, Gap 0.1576 
step 3000: train loss 1.3183, val loss 1.4737, Gap 0.1554 
step 3500: train loss 1.2925, val loss 1.4640, Gap 0.1715 
step 3999: train loss 1.2612, val loss 1.4375, Gap 0.1763 


Ok now we are still improving nicely as the model gets bigger - I think it is worth going to recheck the learning rate - we could see some improvements by stepping down a bit. ( we could also see improvement by decaying learning rate during training). For both of the below tests I ran the above model and trained it over 4000 steps. Then moved to the below config with learning rate lower and trained another 4000 steps. and got these results .

- .0003 got train loss 1.1426, val loss 1.3755, Gap 0.2329
- .0001 got train loss 1.1628, val loss 1.3873, Gap 0.2245

In [79]:
# Training the model further with a lower learning rate ( seeing if we can make a stepwise improvement with lower rate)
learning_rate = .0001
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, Gap {losses['val']-losses['train']:.4f} ")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 1.2683, val loss 1.4437, Gap 0.1754 
step 500: train loss 1.2427, val loss 1.4106, Gap 0.1679 
step 1000: train loss 1.2304, val loss 1.4237, Gap 0.1933 
step 1500: train loss 1.2184, val loss 1.3994, Gap 0.1810 
step 2000: train loss 1.2085, val loss 1.3848, Gap 0.1764 
step 2500: train loss 1.1864, val loss 1.3941, Gap 0.2076 
step 3000: train loss 1.1777, val loss 1.3796, Gap 0.2020 
step 3500: train loss 1.1681, val loss 1.3640, Gap 0.1959 
step 3999: train loss 1.1628, val loss 1.3873, Gap 0.2245 


Ok after those Learning rate tests it seems like there is are small improvements there but the model likely could still be bigger. Lets reset and keep increasing the size of the model.

Here we have now 360 n_embd, 6 n_head, and 8 n_layer lets see how we do (also increasing training at iterations again.
- Tried 6 heads and 8 layers and we improved
- Tried 8 x 8 x 360 and i got the results below
- 12.57347 M parameters
- step 0: train loss 4.8675, val loss 4.8700 Gap 0.0026
- step 1000: train loss 1.5942, val loss 1.7361, Gap 0.1419
- step 2000: train loss 1.3782, val loss 1.5302, Gap 0.1521
- step 3000: train loss 1.2794, val loss 1.4683, Gap 0.1889
- step 4000: train loss 1.2325, val loss 1.4275, Gap 0.1950
- step 4999: train loss 1.2011, val loss 1.3776, Gap 0.1765

After all of that though it seems like this size model needs more time to train however the last run took about 10 minutes so we are starting to run into time constraints.
For the sake of usability I think it makes sense to work with a smaller model but it stands to reason that we could achieve better performance with a larger one.

Moreover the improvement to 1.377 with the modle that is 360 x 8 x 8 is not all that much better than the 360 x 6 x 6 which gave us train loss 1.2612, val loss 1.4375, Gap 0.1763

So for now I think I will stop there with increasing size of the model.



In [82]:
#hyperparameters
batch_size = 32 # how many sequences to process in parallel
block_size = 64 # maximum context length
max_iters = 5000 # total iterations of training
eval_interval = 1000 # how often to evaluate the loss.
learning_rate = .0006 # size of jump during gradient descent.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 360
n_head = 8
n_layer = 8
dropout = 0

In [83]:
# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, Gap {losses['val']-losses['train']:.4f} ")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

12.57347 M parameters
step 0: train loss 4.8675, val loss 4.8700, Gap 0.0026 
step 1000: train loss 1.5942, val loss 1.7361, Gap 0.1419 
step 2000: train loss 1.3782, val loss 1.5302, Gap 0.1521 
step 3000: train loss 1.2794, val loss 1.4683, Gap 0.1889 
step 4000: train loss 1.2325, val loss 1.4275, Gap 0.1950 
step 4999: train loss 1.2011, val loss 1.3776, Gap 0.1765 


OK so we found a good learning rate and have settled on a decent size for our model. Now we are going to look at the context window or "block_size" Currently we have block size set at 64, so a sequence of 64 characters. We are going to move to 128 first to see how that changes things. but to keep things fair we are going to reduce batch size by 1/2 so that we are using the same amount of characters per step.

In [85]:
#hyperparameters
batch_size = 16 # how many sequences to process in parallel
block_size = 128 # maximum context length
max_iters = 5000 # total iterations of training
eval_interval = 1000 # how often to evaluate the loss.
learning_rate = .0006 # size of jump during gradient descent.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 360
n_head = 8
n_layer = 8
dropout = 0

In [86]:
# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, Gap {losses['val']-losses['train']:.4f} ")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

12.59651 M parameters
step 0: train loss 4.7483, val loss 4.7501, Gap 0.0018 
step 1000: train loss 1.6403, val loss 1.7926, Gap 0.1524 
step 2000: train loss 1.3647, val loss 1.5195, Gap 0.1548 
step 3000: train loss 1.2629, val loss 1.4282, Gap 0.1653 
step 4000: train loss 1.2034, val loss 1.3822, Gap 0.1788 
step 4999: train loss 1.1659, val loss 1.3651, Gap 0.1992 


Ok this was a 10 min run of the biggest model yet.

360 x 8 x 8 with batch size of 128 yeilded the following which was our best one yet

12.59651 M parameters
- step 0: train loss 4.7483, val loss 4.7501, Gap 0.0018
- step 1000: train loss 1.6403, val loss 1.7926, Gap 0.1524
- step 2000: train loss 1.3647, val loss 1.5195, Gap 0.1548
- step 3000: train loss 1.2629, val loss 1.4282, Gap 0.1653
- step 4000: train loss 1.2034, val loss 1.3822, Gap 0.1788
- step 4999: train loss 1.1659, val loss 1.3651, Gap 0.1992

OK for some Review stats lets see our progress.

First we established a learning rate of .0006 with results after 1000 steps:
- Train loss 1.5811, val loss 1.7374

Then we bumped the embedding dimension from 128 to 256 to arrive at while training for 2k steps:
- train loss 1.3547, val loss 1.5285

Then we added 2 layers to arrive at n_layers = 6 :
- train loss 1.3590, val loss 1.4756

Then we increased the embedding dimension again to 360 and trained for 4k steps
- train loss 1.2450, val loss 1.4329, Gap 0.1879

Then we added 2 more heads of attention to be 6 with the total model at 360 x 6 x 6 with a batch size of 64 training for 4k steps :
- train loss 1.2612, val loss 1.4375, Gap 0.1763

We checked the learning rate again and did not see much improvement

Then we tried biggest model yet with 360 x 8 x 8
- loss 1.2011, val loss 1.3776, Gap 0.1765

Then we changed the block and batch sizes to 128 and 16 respectively and got.
- train loss 1.1659, val loss 1.3651, Gap 0.1992

In all thought I think our best most efficient model came aroud the 360 x 4 x 6 with 128 block_size

In [89]:
#hyperparameters
batch_size = 8 # how many sequences to process in parallel
block_size = 256 # maximum context length
max_iters = 5000 # total iterations of training
eval_interval = 1000 # how often to evaluate the loss.
learning_rate = .0006 # size of jump during gradient descent.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 50
n_embd = 360
n_head = 4
n_layer = 6
dropout = 0

In [ ]:
# define the model, send it to the device, and print as it runs.
torch.manual_seed(1337) # same start point each time.
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training the model
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, Gap {losses['val']-losses['train']:.4f} ")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

9.52499 M parameters
step 0: train loss 4.5954, val loss 4.5956, Gap 0.0002 
step 1000: train loss 1.9343, val loss 2.0798, Gap 0.1455 
step 2000: train loss 1.5346, val loss 1.7285, Gap 0.1939 
step 3000: train loss 1.3353, val loss 1.4954, Gap 0.1601 


OK so the above model with 360 x 4 x 6 with 128 block size trained in half the time as the biggest model which was 360 x 8 x 8 x 128 and got within .02 of the performance so that is quite good. Now we are going to move on from model size.